# TP Aprendizaje Automático II — Random Forest: baseline y optimización independiente

## Objetivo y alcance

Reproducir el **baseline original de `src/embeddings_palabras.py`** y mejorar el **macro F1 a nivel palabra** de forma **independiente** para:

1. `punt_inicial` (`¿` / sin signo),
2. `punt_final` (`,`, `.`, `?`, sin signo),
3. `capitalizacion` (0, 1, 2, 3).

**Entrada inalterada:** embedding estático de 768 dimensiones por palabra, promedio de embeddings de tokens WordPiece de `bert-base-multilingual-cased`. No se vuelven a calcular los embeddings; se cargan `embeddings_palabras.npy` y `etiquetas_palabras.csv` ya generados. Esto mantiene fijo el pipeline. Los tres bosques son **independientes**; no hay pérdida multitarea.

**Prevención de fuga:** test definido por `GroupShuffleSplit(test_size=0.2, random_state=42)` agrupando por `instancia_id`. La optimización usa una validación interna también separada por instancia (`random_state=2026`). **Nunca elegir candidatos por sus resultados en test.** El baseline original ya había sido evaluado en test en una ejecución anterior; dichos valores se documentan solo como referencia.

**Estado de los experimentos:** resultados medidos en ejecuciones previas se identifican como *históricos*; el código permite reproducirlos. No interpretar el screening reducido como resultado final. La sección de refinamiento ejecuta por tarea modelos de 40 árboles con subtrain completo y su selección. La evaluación final del test es una acción explícita que solo procede tras congelar ganadores.

**Actualización:** se incluyó la búsqueda y validación de umbrales de decisión. La evaluación sobre test permanece desactivada hasta congelar hiperparámetros y umbrales.


## 0. Configuración, dependencias y localización de datos

Se puede ejecutar en un clon del repositorio. Colocar ambos archivos precomputados en el mismo directorio (por ejemplo `salidas/` o `datos_modelo/`) o definir la variable de entorno `RF_DATA_DIR`. **No** se requiere instalar `transformers` para entrenar los bosques. No subir el `.npy` grande a GitHub sin una decisión deliberada de almacenamiento.

In [ ]:
from pathlib import Path
import os, time, json, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED_TEST = 42
SEED_VALIDACION = 2026
SEED_MODELO = 42
TAREAS = ["punt_inicial", "punt_final", "capitalizacion"]

# Para usar este notebook en el repositorio, configurar si es necesario:
# os.environ["RF_DATA_DIR"] = "/ruta/al/directorio/con/embeddings"
ubicaciones = [Path(os.environ.get("RF_DATA_DIR", ""))] if os.environ.get("RF_DATA_DIR") else []
ubicaciones += [Path.cwd(), Path.cwd() / "salidas", Path.cwd() / "datos_modelo",
               Path.cwd().parent, Path.cwd().parent / "salidas",
               Path("/mnt/data"), Path("/content/drive/MyDrive/TP_AA")]
DATA_DIR = next((p for p in ubicaciones if (p/"embeddings_palabras.npy").is_file()
                 and (p/"etiquetas_palabras.csv").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Faltan embeddings_palabras.npy y etiquetas_palabras.csv. Definir RF_DATA_DIR.")
OUTPUT_DIR = Path("resultados_rf")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
print("Datos:", DATA_DIR.resolve(), "| Resultados:", OUTPUT_DIR.resolve())

## 1. Cargar datos y revisar correspondencia entre filas

**Importantísimo:** la matriz y las etiquetas deben provenir de la **misma ejecución** de generación, y sus filas deben preservar exactamente el mismo orden. El `.npy` se abre con `mmap_mode='r'` para reducir memoria.

In [ ]:
X = np.load(DATA_DIR / "embeddings_palabras.npy", mmap_mode="r")
y = pd.read_csv(DATA_DIR / "etiquetas_palabras.csv", keep_default_na=False)
assert X.ndim == 2 and X.shape[1] == 768
assert X.shape[0] == len(y)
assert set(["instancia_id", *TAREAS]).issubset(y.columns)
assert np.issubdtype(X.dtype, np.floating)
print("X", X.shape, X.dtype, "| etiquetas", y.shape)
display(y.head())
for tarea in TAREAS:
    print("\n", tarea)
    display(y[tarea].value_counts(dropna=False).to_frame("frecuencia")
            .assign(porcentaje=lambda df: 100*df.frecuencia/len(y)))

## 2. Splits por fragmento: train/test original + subtrain/validación

La selección **no** utiliza etiquetas del test. Este split interno se fija para todas las configuraciones, garantizando comparaciones pareadas. Nota: las palabras idénticas pueden aparecer en diferentes fragmentos; el split evita compartir fragmentos, **no** garantiza separar vocabulario.

In [ ]:
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=SEED_TEST)
                           .split(y, groups=y["instancia_id"]))
sub_rel, val_rel = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=SEED_VALIDACION)
                        .split(y.iloc[train_idx], groups=y.iloc[train_idx]["instancia_id"]))
subtrain_idx = train_idx[sub_rel]
val_idx = train_idx[val_rel]
assert set(y.iloc[train_idx].instancia_id).isdisjoint(y.iloc[test_idx].instancia_id)
assert set(y.iloc[subtrain_idx].instancia_id).isdisjoint(y.iloc[val_idx].instancia_id)
print({"train": len(train_idx), "subtrain": len(subtrain_idx),
       "validacion":len(val_idx), "test":len(test_idx)})
np.savez(OUTPUT_DIR/"splits.npz", train=train_idx, test=test_idx,
         subtrain=subtrain_idx, validacion=val_idx)

## 3. Baseline original — tres RF de 100 árboles

Este bloque reproduce la configuración original del repositorio: `RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42)`, un bosque por tarea, entrenado sobre el **train original** y evaluado en **test**. **No** usar estos puntajes para elegir hiperparámetros.

La ejecución original ya se realizó; los siguientes resultados corresponden a esa ejecución y se preservan aquí para documentación (no salen de una corrida nueva de esta celda):

In [ ]:
baseline_historico = pd.DataFrame([
    {"tarea":"punt_inicial", "accuracy":0.8422492220482429, "macro_f1":0.5249340301644748, "weighted_f1":0.9009579963335721},
    {"tarea":"punt_final", "accuracy":0.6358591164053674, "macro_f1":0.3576778402143853, "weighted_f1":0.7085309357604083},
    {"tarea":"capitalizacion", "accuracy":0.7613917903791647, "macro_f1":0.4283128785151419, "weighted_f1":0.8016505878148146},
])
display(baseline_historico)
# Reproducción opcional: ~400 segundos en el entorno donde se midió originalmente.
REPRODUCIR_BASELINE = False
if REPRODUCIR_BASELINE:
    X_train = np.asarray(X[train_idx]); X_test = np.asarray(X[test_idx])
    baseline_actual = []
    for tarea in TAREAS:
        inicio = time.monotonic()
        modelo = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                         n_jobs=-1, random_state=SEED_MODELO)
        modelo.fit(X_train, y[tarea].to_numpy()[train_idx])
        pred = modelo.predict(X_test)
        verdad = y[tarea].to_numpy()[test_idx]
        baseline_actual.append({"tarea": tarea,
                    "accuracy": accuracy_score(verdad,pred),
                    "macro_f1": f1_score(verdad,pred,average="macro",zero_division=0),
                    "weighted_f1":f1_score(verdad,pred,average="weighted",zero_division=0),
                    "segundos":round(time.monotonic()-inicio,1)})
        print(tarea,classification_report(verdad,pred,zero_division=0))
        pd.DataFrame(confusion_matrix(verdad,pred,labels=modelo.classes_),
                     index=modelo.classes_,columns=modelo.classes_).to_csv(OUTPUT_DIR/f"baseline_confusion_{tarea}.csv")
    pd.DataFrame(baseline_actual).to_csv(OUTPUT_DIR/"baseline_reproducido.csv",index=False)
    display(pd.DataFrame(baseline_actual))

## 4. Screening rápido — 12 configuraciones ejecutadas

Objetivo: orientar la búsqueda barata antes de entrenar sobre subtrain completo. Se eligen grupos completos (~12 % de los fragmentos del subtrain), **12 árboles**, `max_samples=.75`, `max_features='sqrt'`, 2 núcleos. **No** comparar este F1 directamente con el baseline en test: diferentes datos de entrenamiento/validación y menor cantidad de árboles.

Se incluyen los resultados **observados** en la exploración previa para poder inspeccionarlos sin recalcular. La celda siguiente permite reproducirlos cuando se desee.

In [ ]:
from io import StringIO
screening_historico = pd.read_csv(StringIO('tarea,candidate,macro_f1_val,accuracy_val,seconds,screen_n,val_n\npunt_inicial,none_depth12,0.583540424337834,0.9845380511784346,2.5,38027,81167\npunt_inicial,balanced_depth12,0.5309860893599765,0.85989379920411,1.8,38027,81167\npunt_inicial,none_depth20,0.594675451989417,0.9844518092328164,3.2,38027,81167\npunt_inicial,balanced_depth12_leaf3,0.5313275495297749,0.859918439760001,1.7,38027,81167\npunt_final,none_depth12,0.2886784073355128,0.845959564847783,2.2,38027,81167\npunt_final,balanced_depth12,0.337420011233763,0.618330109527271,2.0,38027,81167\npunt_final,none_depth20,0.3187726758585689,0.8384934764128279,2.8,38027,81167\npunt_final,balanced_depth12_leaf3,0.3288724440534555,0.5893528157995245,2.1,38027,81167\ncapitalizacion,none_depth12,0.3016142101099771,0.8980250594453411,2.1,38027,81167\ncapitalizacion,balanced_depth12,0.3721451259544421,0.7508593393866966,1.7,38027,81167\ncapitalizacion,none_depth20,0.4364361079503202,0.9002796703093622,2.6,38027,81167\ncapitalizacion,balanced_depth12_leaf3,0.3732455201543603,0.7454260968127441,1.7,38027,81167\n'))
display(screening_historico.pivot(index="candidate", columns="tarea",values="macro_f1_val").round(4))
screening_historico.to_csv(OUTPUT_DIR/"screening_historico.csv",index=False)

REPRODUCIR_SCREENING = False
if REPRODUCIR_SCREENING:
    grupos = np.unique(y.iloc[subtrain_idx]["instancia_id"])
    seleccionados = np.random.default_rng(2026).choice(grupos,size=int(len(grupos)*.12),replace=False)
    idx_screen = subtrain_idx[np.isin(y.iloc[subtrain_idx]["instancia_id"], seleccionados)]
    X_screen=np.asarray(X[idx_screen]); X_val=np.asarray(X[val_idx])
    configs_screen={
        "none_depth12": dict(class_weight=None,max_depth=12,min_samples_leaf=1),
        "balanced_depth12": dict(class_weight="balanced",max_depth=12,min_samples_leaf=1),
        "none_depth20": dict(class_weight=None,max_depth=20,min_samples_leaf=1),
        "balanced_depth12_leaf3":dict(class_weight="balanced",max_depth=12,min_samples_leaf=3),
    }
    filas=[]
    for tarea in TAREAS:
        etiquetas=y[tarea].to_numpy()
        for nombre,parametros in configs_screen.items():
            rf=RandomForestClassifier(n_estimators=12,max_features="sqrt",max_samples=.75,
                                       n_jobs=2,random_state=SEED_MODELO,**parametros)
            rf.fit(X_screen,etiquetas[idx_screen]); pred=rf.predict(X_val)
            filas.append(dict(tarea=tarea,candidate=nombre,
               macro_f1_val=f1_score(etiquetas[val_idx],pred,average="macro",zero_division=0),
               accuracy_val=accuracy_score(etiquetas[val_idx],pred)))
    pd.DataFrame(filas).to_csv(OUTPUT_DIR/"screening_reproducido.csv",index=False)
    display(pd.DataFrame(filas))

## 5. Refinamiento: subtrain completo, modelos independientes

La selección ahora se hace con **318.900** palabras del subtrain y la misma validación de **81.167** palabras, 40 árboles por candidato. Una comparación tiene sentido **dentro de esta etapa y la misma tarea**. El número de árboles es igual entre competidores.

Shortlist conservadora informada por el screening: comparación entre pesos automáticos y sin pesos con profundidad 20 en inicial/capitalización; dos profundidades para puntuación final con `balanced`.

Aquí los hiperparámetros se optimizan **por tarea**; los futuros pasos pueden ampliar `min_samples_leaf`, `max_features`, `max_samples`, `balanced_subsample`, pesos manuales y 100+ árboles. El screening no prueba exhaustivamente estos valores.

La celda guarda resultados incrementalmente y permite reanudar después de una interrupción.

In [ ]:
CANDIDATOS_REFINAMIENTO = {
    "punt_inicial": {
        "none_depth20": dict(class_weight=None,max_depth=20,min_samples_leaf=1),
        "balanced_depth20": dict(class_weight="balanced",max_depth=20,min_samples_leaf=1),
    },
    "punt_final": {
        "balanced_depth12": dict(class_weight="balanced",max_depth=12,min_samples_leaf=1),
        "balanced_depth20": dict(class_weight="balanced",max_depth=20,min_samples_leaf=1),
    },
    "capitalizacion": {
        "none_depth20": dict(class_weight=None,max_depth=20,min_samples_leaf=1),
        "balanced_depth20": dict(class_weight="balanced",max_depth=20,min_samples_leaf=1),
    },
}
N_ARBOLES_REFINAMIENTO = 40
REFINAMIENTO_CSV = OUTPUT_DIR/"refinamiento_40_arboles.csv"

# Seleccionar True para reproducir / continuar el refinamiento; requiere tiempo y RAM.
EJECUTAR_REFINAMIENTO = False
if EJECUTAR_REFINAMIENTO:
    df_existe = pd.read_csv(REFINAMIENTO_CSV) if REFINAMIENTO_CSV.exists() else pd.DataFrame()
    ya = set(zip(df_existe.get('tarea',[]),df_existe.get('candidate',[])))
    filas = df_existe.to_dict('records')
    X_sub = np.asarray(X[subtrain_idx]); X_val = np.asarray(X[val_idx])
    for tarea, candidatos in CANDIDATOS_REFINAMIENTO.items():
        yy = y[tarea].to_numpy()
        for nombre, parametros in candidatos.items():
            if (tarea,nombre) in ya:
                continue
            inicio = time.monotonic()
            rf = RandomForestClassifier(n_estimators=N_ARBOLES_REFINAMIENTO, max_features='sqrt',
                                         n_jobs=2,random_state=SEED_MODELO,**parametros)
            rf.fit(X_sub,yy[subtrain_idx]); pred=rf.predict(X_val)
            resultado=dict(tarea=tarea,candidate=nombre,n_estimators=N_ARBOLES_REFINAMIENTO,
                   class_weight=str(parametros['class_weight']),max_depth=parametros['max_depth'],
                   min_samples_leaf=parametros['min_samples_leaf'],max_features='sqrt',
                   macro_f1_val=f1_score(yy[val_idx],pred,average='macro',zero_division=0),
                   accuracy_val=accuracy_score(yy[val_idx],pred),
                   elapsed_seconds=round(time.monotonic()-inicio,1),
                   subtrain_n=len(subtrain_idx),val_n=len(val_idx))
            filas.append(resultado)
            pd.DataFrame(filas).to_csv(REFINAMIENTO_CSV,index=False)
            (OUTPUT_DIR/f'reporte_refinado_{tarea}_{nombre}.txt').write_text(
                classification_report(yy[val_idx],pred,zero_division=0),encoding='utf-8')
            print('Guardado:',resultado)
    display(pd.DataFrame(filas))

### Resultados del refinamiento observado

La siguiente celda carga los resultados medidos **si están disponibles**. Al subir al repositorio, se puede copiar el CSV generado a `resultados_rf/` para que la tabla aparezca sin reentrenar. No se inventan valores de candidatos no ejecutados.

In [ ]:
# Resultados históricos efectivamente medidos (6 corridas sobre subtrain completo).
# Se incorporan al notebook para poder inspeccionarlos aunque no esté presente el CSV.
from io import StringIO
refinamiento_historico = pd.read_csv(StringIO('tarea,candidate,n_estimators,class_weight,max_depth,min_samples_leaf,max_features,macro_f1_val,accuracy_val,elapsed_seconds,subtrain_n,val_n\npunt_inicial,none_depth20,40,None,20,1,sqrt,0.5967076591436274,0.9844518092328164,95.6,318900,81167\npunt_inicial,balanced_depth20,40,balanced,20,1,sqrt,0.5256481990664569,0.8442100853795261,64.7,318900,81167\npunt_final,balanced_depth12,40,balanced,12,1,sqrt,0.3335202305609455,0.5560634247908632,61.4,318900,81167\npunt_final,balanced_depth20,40,balanced,20,1,sqrt,0.3494960290950546,0.6027326376483053,79.5,318900,81167\ncapitalizacion,none_depth20,40,None,20,1,sqrt,0.5510665563502042,0.9096923626597015,77.6,318900,81167\ncapitalizacion,balanced_depth20,40,balanced,20,1,sqrt,0.4373409693698573,0.7689701479665382,62.0,318900,81167\n'))
if REFINAMIENTO_CSV.exists():
    refinado = pd.read_csv(REFINAMIENTO_CSV, keep_default_na=False)
else:
    refinado = refinamiento_historico.copy()
    refinado.to_csv(REFINAMIENTO_CSV,index=False)
display(refinado.sort_values(['tarea','macro_f1_val'],ascending=[True,False]))

### Interpretación de las seis corridas (resultados medidos)

| Tarea | Candidato | Macro F1 validación | Accuracy validación |
|---|---|---:|---:|
| Inicial | Sin pesos, profundidad 20 | **0,5967** | 0,9845 |
| Inicial | Balanceado, profundidad 20 | 0,5256 | 0,8442 |
| Final | Balanceado, profundidad 12 | 0,3335 | 0,5561 |
| Final | Balanceado, profundidad 20 | **0,3495** | 0,6027 |
| Capitalización | Sin pesos, profundidad 20 | **0,5511** | 0,9097 |
| Capitalización | Balanceado, profundidad 20 | 0,4373 | 0,7690 |

**Advertencia sobre el desbalance:** en puntuación inicial, sin pesos se logra precision ~0,52 y recall ~0,12 para `¿`; con pesos balanceados, precision ~0,07 y recall ~0,79. Aunque la primera configuración da mejor macro F1, **se pierden la mayoría de los signos `¿`**. Esto amerita explorar pesos intermedios y umbrales de decisión calibrados en validación (nunca en test).

## 6. Elección de mejores hiperparámetros *por tarea* en validación

Los ganadores se eligen por macro F1 sobre validación, **sin tocar test**. Se registra accuracy y métricas por clase por separado para detectar colapso hacia clases frecuentes. La selección se condiciona a que cada tarea tenga completados todos sus candidatos; si no, no debe considerarse definitiva.

In [ ]:
ganadores = {}
if not refinado.empty:
    for tarea in TAREAS:
        subset = refinado.loc[refinado.tarea==tarea]
        previstos=set(CANDIDATOS_REFINAMIENTO[tarea])
        vistos=set(subset.candidate)
        if not previstos.issubset(vistos):
            print('INCOMPLETO',tarea,'faltan:',previstos-vistos)
            continue
        fila = subset.loc[subset.macro_f1_val.idxmax()]
        ganadores[tarea] = dict(CANDIDATOS_REFINAMIENTO[tarea][fila.candidate])
        print(tarea,'ganador:',fila.candidate,'F1 validación:',round(fila.macro_f1_val,5))
    if len(ganadores)==len(TAREAS):
        (OUTPUT_DIR/'ganadores.json').write_text(json.dumps(ganadores,ensure_ascii=False,indent=2))
else:
    print('Todavía no hay resultados de refinamiento cargados.')

## 7. Ajuste de umbrales de decisión en validación (sin reentrenar el bosque)

**Objetivo:** optimizar macro F1 por tarea en las **81.167 palabras de validación**; el conjunto de test original (100.906 palabras) no interviene en esta fase. Partimos de la configuración ganadora de la ronda anterior de cada Random Forest (40 árboles, profundidad máxima 20; `class_weight=None` para `punt_inicial` y `capitalizacion`, `balanced` para `punt_final`).

- En `punt_inicial`, barrer el umbral para la probabilidad de la clase positiva (`¿`). Un umbral bajo incrementa recall pero también los falsos positivos.
- En `punt_final` y `capitalizacion`, `predict_proba` devuelve una probabilidad por clase. La regla estándar `argmax` se cambia por `argmax(probabilidad * multiplicador_por_clase)`; se buscan multiplicadores positivos mediante búsqueda por coordenadas. **Estos multiplicadores no son `class_weight` de entrenamiento**; actúan solo después de obtener las probabilidades.
- La búsqueda de umbrales se hace sobre la misma validación que ayudó a seleccionar hiperparámetros. Su macro F1 queda sujeto a sesgo optimista por selección repetida; para una estimación menos sesgada convendría otra partición de calibración o una validación interna adicional.
- Importante: las probabilidades del RF no necesariamente están calibradas. Este procedimiento mejora decisiones empíricamente en validación, no garantiza calibración de probabilidades.
- **Test reservado.** No elegir umbrales mirando resultados de test.


In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

PARAMS_UMBRAL = {
    'punt_inicial': dict(class_weight=None, max_depth=20, n_estimators=40),
    'punt_final': dict(class_weight='balanced', max_depth=20, n_estimators=40),
    'capitalizacion': dict(class_weight=None, max_depth=20, n_estimators=40),
}

# Mantener X en mmap antes de construir las matrices de subtrain y validación.
# Este código puede necesitar >3 GB de RAM durante los entrenamientos.

def optimizar_probabilidades(proba, clases, y_true):
    """Elige parámetros post-hoc en validación para maximizar macro F1."""
    clases = np.asarray(clases)
    verdadero = np.searchsorted(clases, np.asarray(y_true))
    n_clases = len(clases)
    clasif_default = proba.argmax(axis=1)
    base = f1_score(verdadero, clasif_default, average='macro', labels=np.arange(n_clases), zero_division=0)
    trazas = []
    if n_clases == 2:
        pos = int(np.flatnonzero(clases == '¿')[0])
        neg = 1-pos
        candidatos = np.unique(np.r_[np.linspace(.005,.995,199),.001,.002,.003,.999])
        mejor = (-1.0, None, None)
        for umbral in candidatos:
            pred = np.where(proba[:, pos] >= umbral, pos, neg)
            score = f1_score(verdadero, pred, average='macro', labels=np.arange(n_clases), zero_division=0)
            trazas.append(dict(parametro='umbral_¿', valor=umbral, macro_f1=score))
            if score > mejor[0]: mejor = (score, float(umbral), pred)
        config = {'metodo':'umbral_binario', 'clase_positiva':'¿', 'umbral':mejor[1]}
        pred_optim = mejor[2]
    else:
        pesos = np.ones(n_clases)
        conteos = np.bincount(verdadero, minlength=n_clases)
        mayoritaria = int(np.argmax(conteos))
        candidatos = [.05,.1,.15,.2,.3,.4,.5,.65,.8,1,1.25,1.5,2,2.5,3,4,5,7,10,15,20,30,50]
        mejor_score = base
        for vuelta in range(4):
            cambios = 0
            for j in np.argsort(conteos):
                if j == mayoritaria: continue
                nuevo_peso = pesos[j]
                for f in candidatos:
                    prueba = pesos.copy(); prueba[j] = f
                    pred = np.argmax(proba * prueba, axis=1)
                    score = f1_score(verdadero, pred, average='macro', labels=np.arange(n_clases), zero_division=0)
                    trazas.append(dict(parametro=str(clases[j]), valor=f, macro_f1=score, vuelta=vuelta))
                    if score > mejor_score + 1e-10:
                        nuevo_peso = f
                        mejor_score = score
                if nuevo_peso != pesos[j]:
                    pesos[j] = nuevo_peso
                    cambios += 1
            if cambios == 0: break
        pred_optim = np.argmax(proba * pesos, axis=1)
        config = {'metodo':'multiplicadores_multiclase', 'multiplicadores':{str(k):float(v) for k,v in zip(clases,pesos)}}
    return clasif_default, pred_optim, config, pd.DataFrame(trazas)


In [ ]:
# Reproduce los entrenamientos de las configuraciones refinadas y calcula probabilidades
# para buscar umbrales en validación. Los archivos CSV/JSON también pueden cargarse
# de una ejecución anterior si están presentes.
EJECUTAR_AJUSTE_UMBRAL = False  # Cambiar a True para repetir las tres corridas
RUTA_RES_UMBRAL = OUTPUT_DIR / 'resultados_umbrales.csv'
RUTA_PARAM_UMBRAL = OUTPUT_DIR / 'umbrales_seleccionados.json'

if EJECUTAR_AJUSTE_UMBRAL:
    import gc
    x_subtrain = np.asarray(X[subtrain_idx]); x_val = np.asarray(X[val_idx])
    resultados = []; configs = {}; curvas = []
    for tarea, pars in PARAMS_UMBRAL.items():
        rf = RandomForestClassifier(
            n_estimators=pars['n_estimators'], max_depth=pars['max_depth'],
            class_weight=pars['class_weight'], min_samples_leaf=1,
            max_features='sqrt', random_state=42, n_jobs=2)
        rf.fit(x_subtrain, y[tarea].to_numpy()[subtrain_idx])
        prob = rf.predict_proba(x_val)
        pred0, pred1, config, curva = optimizar_probabilidades(prob, rf.classes_, y[tarea].to_numpy()[val_idx])
        verdad = np.searchsorted(rf.classes_, y[tarea].to_numpy()[val_idx])
        clases = np.arange(len(rf.classes_))
        score0 = f1_score(verdad,pred0,average='macro',labels=clases,zero_division=0)
        score1 = f1_score(verdad,pred1,average='macro',labels=clases,zero_division=0)
        resultados.append(dict(tarea=tarea,macro_f1_default=score0,macro_f1_ajustado=score1,delta_f1=score1-score0,
            accuracy_default=accuracy_score(verdad,pred0),accuracy_ajustado=accuracy_score(verdad,pred1)))
        configs[tarea] = config
        curvas.append(curva.assign(tarea=tarea))
        for nombre,pred in [('default',pred0),('ajustado',pred1)]:
            (OUTPUT_DIR / f'reporte_umbrales_{tarea}_{nombre}.txt').write_text(
                classification_report(verdad,pred,labels=clases,target_names=[repr(x) for x in rf.classes_],zero_division=0),encoding='utf-8')
        del rf,prob;gc.collect()
    pd.DataFrame(resultados).to_csv(RUTA_RES_UMBRAL,index=False)
    pd.concat(curvas).to_csv(OUTPUT_DIR/'exploracion_umbrales.csv',index=False)
    RUTA_PARAM_UMBRAL.write_text(json.dumps(configs,indent=2,ensure_ascii=False),encoding='utf-8')
# Resultados medidos en esta conversación, incluidos para poder revisar el notebook
# aun cuando todavía no se hayan copiado los CSV de resultados al repositorio.
from io import StringIO
historico_umbrales = pd.read_csv(StringIO('tarea,n_estimators,max_depth,class_weight,n_subtrain,n_val,macro_f1_default,macro_f1_ajustado,delta_f1,accuracy_default,accuracy_ajustado,time_seconds\npunt_inicial,40,20,,318900,81167,0.5967076591436274,0.6935439850574989,0.0968363259138714,0.9844518092328164,0.9810760530757574,103.6\npunt_final,40,20,balanced,318900,81167,0.3494960290950546,0.4032713637831734,0.0537753346881187,0.6027326376483053,0.7854300393016866,81.2\ncapitalizacion,40,20,,318900,81167,0.5510665563502042,0.5977233436013004,0.0466567872510962,0.9096923626597015,0.9020168294996735,80.7\n'))
historico_config_umbrales = {'punt_inicial': {'method': 'binary_probability_threshold', 'positive_class': '¿', 'threshold': 0.195}, 'punt_final': {'method': 'multiclass_probability_multipliers', 'class_multipliers': {'': 1.0, ',': 0.4, '.': 0.4, '?': 0.15}, 'majority_class': ''}, 'capitalizacion': {'method': 'multiclass_probability_multipliers', 'class_multipliers': {'0': 1.0, '1': 2.5, '2': 3.0, '3': 4.0}, 'majority_class': '0'}}
if RUTA_RES_UMBRAL.exists():
    resultados_umbrales = pd.read_csv(RUTA_RES_UMBRAL)
else:
    resultados_umbrales = historico_umbrales.copy()
    resultados_umbrales.to_csv(RUTA_RES_UMBRAL,index=False)
if not RUTA_PARAM_UMBRAL.exists():
    RUTA_PARAM_UMBRAL.write_text(json.dumps(historico_config_umbrales,ensure_ascii=False,indent=2),encoding='utf-8')
if True:
    display(resultados_umbrales)
    if RUTA_PARAM_UMBRAL.exists():
        umbrales_elegidos=json.loads(RUTA_PARAM_UMBRAL.read_text(encoding='utf-8'))
        print(json.dumps(umbrales_elegidos,indent=2,ensure_ascii=False))
else:
    print('Los resultados aún no están presentes. Ejecutar la celda con EJECUTAR_AJUSTE_UMBRAL=True.')


### Resultados observados de ajuste post-hoc (sobre validación)

Los siguientes valores **fueron medidos** entrenando nuevamente los tres RF con las configuraciones ganadoras del refinamiento. Se usaron **las mismas** 81.167 palabras de validación, sin ninguna muestra del test. El ajuste se realiza únicamente sobre las probabilidades de salida.

| Tarea | Macro F1 antes | Macro F1 después | Ganancia | Accuracy antes → después |
|---|---:|---:|---:|---:|
| Inicial | 0,5967 | **0,6935** | +0,0968 | 98,45% → 98,11% |
| Final | 0,3495 | **0,4033** | +0,0538 | 60,27% → 78,54% |
| Capitalización | 0,5511 | **0,5977** | +0,0467 | 90,97% → 90,20% |

**Parámetros elegidos mediante validación:**

- Puntuación inicial: `P(¿) >= 0.195`.
- Puntuación final: multiplicar las probabilidades de las clases `''`, `','`, `'.'`, `'?'` por `1.0`, `0.4`, `0.4`, `0.15`, respectivamente, y tomar `argmax`.
- Capitalización: multiplicar las probabilidades de las clases `0`, `1`, `2`, `3` por `1.0`, `2.5`, `3.0`, `4.0`, respectivamente, y tomar `argmax`.

**Clases minoritarias:** en puntuación inicial, el nuevo modelo encuentra 505 de 1.277 signos `¿` (recall ~39,5%, precision ~39,8%); no se trata de una solución perfecta. La macro F1 de validación puede estar sobreajustada debido al proceso de selección; **no extrapolar estas ganancias al test sin evaluación final**.


In [ ]:
# Visualización de mejoras en validación
fig, ax = plt.subplots(figsize=(8,4))
tareas = resultados_umbrales['tarea'].tolist()
pos = np.arange(len(tareas)); ancho = .38
ax.bar(pos-ancho/2, resultados_umbrales.macro_f1_default, width=ancho, label='Predicción estándar')
ax.bar(pos+ancho/2, resultados_umbrales.macro_f1_ajustado, width=ancho, label='Regla ajustada')
ax.set_xticks(pos,tareas); ax.set_ylabel('Macro F1 (validación)')
ax.set_title('Mismo RF: cambio exclusivo en regla de decisión')
ax.legend(); fig.tight_layout()
fig.savefig(OUTPUT_DIR/'comparacion_macro_f1_umbrales.png',dpi=140)
plt.show()


### Interpretación de la calibración de decisiones

- **Comparación justa:** `macro_f1_default` y `macro_f1_ajustado` provienen de **los mismos bosques** y **las mismas etiquetas de validación**; la única diferencia es la regla de decisión.
- Un aumento de macro F1 puede acompañarse de una **caída de accuracy** por predecir más clases minoritarias. Inspeccionar siempre precision y recall por clase en los reportes.
- Los parámetros post-hoc se guardan en `umbrales_seleccionados.json` y deben aplicarse sin volver a ajustarlos sobre test.
- La etapa de test final debe entrenar de nuevo cada RF sobre **train completo** (subtrain + validación) y aplicar directamente la regla de decisión congelada. Si se cambia el número de árboles u otro parámetro, esa variante debe validarse antes.


## 8. Evaluación final — después de congelar ganadores

**No ejecutar hasta completar y congelar la selección**. Para cada tarea, entrenar con **TODO el train original** y evaluar **una sola vez** sobre test. Para comparar justamente con el baseline, mantener la misma definición de etiquetas/embeddings/split; el candidato ganador se evalúa con **40 árboles**, la misma cantidad con la que fue validado. Si se quieren 100 árboles, primero comparar esa variante en validación y congelarla, sin usar test para elegir.

Se guardan reportes por clase y matrices de confusión. Al repetir búsquedas futuras, **no** reutilizar los puntajes test para seleccionar más configuraciones; en ese caso haría falta un conjunto final verdaderamente nuevo.

In [ ]:
# ¡SOLO DESPUÉS DE CONGELAR TODAS LAS CONFIGURACIONES Y UMBRALES!
EVALUAR_TEST_FINAL = False  # Ya fue ejecutado una sola vez; activar para reproducirlo
N_ARBOLES_FINALES = 40
if EVALUAR_TEST_FINAL:
    if len(ganadores) != len(TAREAS):
        raise RuntimeError('No están seleccionados los tres RF')
    if not RUTA_PARAM_UMBRAL.exists():
        raise RuntimeError('Falta umbrales_seleccionados.json')
    reglas = json.loads(RUTA_PARAM_UMBRAL.read_text(encoding='utf-8'))
    if set(reglas) != set(TAREAS):
        raise RuntimeError('Faltan reglas de decisión para alguna tarea')
    X_train = np.asarray(X[train_idx]); X_test = np.asarray(X[test_idx])
    resultados_test = []; predicciones = {}
    for tarea, parametros in ganadores.items():
        rf = RandomForestClassifier(n_estimators=N_ARBOLES_FINALES,max_features='sqrt',
                                    n_jobs=2,random_state=SEED_MODELO,**parametros)
        rf.fit(X_train, y[tarea].to_numpy()[train_idx])
        proba = rf.predict_proba(X_test)
        regla = reglas[tarea]
        if regla['method'] in ['binary_probability_threshold','umbral_binario']:
            pos = int(np.flatnonzero(rf.classes_ == regla.get('positive_class', regla.get('clase_positiva')))[0])
            negativo = 1 - pos
            pred = np.where(proba[:,pos] >= regla.get('threshold',regla.get('umbral')),rf.classes_[pos],rf.classes_[negativo])
        else:
            pesos = regla.get('class_multipliers', regla.get('multiplicadores'))
            w = np.asarray([pesos[str(clase)] for clase in rf.classes_])
            pred = rf.classes_[np.argmax(proba*w,axis=1)]
        verdad = y[tarea].to_numpy()[test_idx]
        predicciones[tarea] = pred
        resultados_test.append(dict(tarea=tarea,accuracy=accuracy_score(verdad,pred),
            macro_f1=f1_score(verdad,pred,average='macro',zero_division=0),
            weighted_f1=f1_score(verdad,pred,average='weighted',zero_division=0)))
        (OUTPUT_DIR/f'reporte_test_ajustado_{tarea}.txt').write_text(classification_report(verdad,pred,zero_division=0),encoding='utf-8')
        pd.DataFrame(confusion_matrix(verdad,pred,labels=rf.classes_),
            index=[repr(z) for z in rf.classes_],columns=[repr(z) for z in rf.classes_]).to_csv(
            OUTPUT_DIR/f'confusion_test_ajustado_{tarea}.csv')
        np.save(OUTPUT_DIR/f'pred_test_ajustado_{tarea}.npy',pred)
    pd.DataFrame(resultados_test).to_csv(OUTPUT_DIR/'metricas_test_ajustado.csv',index=False)
    print('Accuracy conjunto:',np.mean(np.logical_and.reduce([
        predicciones[t]==y[t].to_numpy()[test_idx] for t in TAREAS])))
    display(pd.DataFrame(resultados_test).merge(baseline_historico,on='tarea',suffixes=('_nuevo','_baseline')))


## 9. Resultados del test final reservado — evaluación única

Se congelaron las configuraciones según **macro F1 de validación** (sin usar etiquetas de test), y luego se ajustaron tres Random Forest sobre las **400.067 palabras** del train original, usando `random_state=42`, `max_depth=20`, `n_estimators=40`, `max_features="sqrt"`, `min_samples_leaf=1`, y `class_weight` específico por tarea. Se comparó el argmax predeterminado contra las reglas de decisión fijas seleccionadas previamente en validación.

- **Puntuación inicial**: `class_weight=None`, umbral `P(¿) >= 0.195`.
- **Puntuación final**: `class_weight="balanced"`, factores `{sin signo: 1, coma: 0.4, punto: 0.4, ?: 0.15}`.
- **Capitalización**: `class_weight=None`, factores `{0: 1, 1: 2.5, 2: 3, 3: 4}`.

Los resultados observados aparecen debajo y se guardan también en CSV. **No se realizó ninguna búsqueda posterior ni ajuste mirando el test.** El baseline usa otro RF de 100 árboles sin límite de profundidad; la comparación mide el pipeline completo, no exclusivamente el efecto de los umbrales.


In [ ]:
# Resultados medidos con el test reservado (se pueden consultar sin reentrenar).
from io import StringIO
resultados_test_final = pd.read_csv(StringIO('tarea,regla,accuracy,macro_f1,weighted_f1,n_train,n_test,n_estimators,max_depth,class_weight\npunt_inicial,default,0.9839454541850832,0.590946189418053,0.9792567686409795,400067,100906,40,20,\npunt_inicial,ajustado,0.98013993221414,0.6766589764992283,0.9800405910800856,400067,100906,40,20,\npunt_final,default,0.597476859651557,0.3497717688448294,0.6820045979553877,400067,100906,40,20,balanced\npunt_final,ajustado,0.7830654272293025,0.402885008411971,0.7926068171555731,400067,100906,40,20,balanced\ncapitalizacion,default,0.908935048460944,0.5885584052148273,0.8856204186944612,400067,100906,40,20,\ncapitalizacion,ajustado,0.9011951717439992,0.6309747645674684,0.892230127444454,400067,100906,40,20,\n'))
comparacion_test_final = pd.read_csv(StringIO('tarea,baseline_macro_f1,baseline_accuracy,rf40_ajustado_macro_f1,rf40_default_macro_f1,rf40_ajustado_accuracy,delta_f1_vs_baseline\npunt_inicial,0.5249340301644748,0.8422492220482429,0.6766589764992283,0.590946189418053,0.98013993221414,0.15172494633475353\npunt_final,0.3576778402143853,0.6358591164053674,0.402885008411971,0.3497717688448294,0.7830654272293025,0.04520716819758569\ncapitalizacion,0.4283128785151419,0.7613917903791647,0.6309747645674684,0.5885584052148273,0.9011951717439992,0.2026618860523265\n'))
display(resultados_test_final[['tarea','regla','accuracy','macro_f1','weighted_f1']])
display(comparacion_test_final)
print('Accuracy conjunto (decisión ajustada):', 0.7027728777277863)


## 10. Interpretación y próximos barridos

- **Objetivo principal:** macro F1 por tarea (promedio no ponderado entre sus clases), sensible a minoritarias.
- **Métrica secundaria:** accuracy, weighted F1, precision/recall y matrices de confusión; accuracy alto puede coexistir con recall muy bajo de `¿` / `?`.
- `class_weight='balanced'` corrige pesos **durante el entrenamiento**, pero no está garantizado que maximice macro F1; probar también `balanced_subsample`, `None` y pesos manuales con validación.
- Próximo barrido recomendado, sin mirar test: `n_estimators=[40,100,200]`, `max_depth=[12,20,30,None]`, `min_samples_leaf=[1,2,5]`, `max_features=['sqrt',0.2,0.5]` y distintas ponderaciones, con `RandomizedSearchCV` o successive halving **por tarea** y CV agrupada por instancia, cuidando los costos.
- Los resultados de este notebook son propios de estos embeddings **estáticos y sin contexto**; cambiar embeddings o incorporar vecinos sería un experimento de pipeline diferente.

**Trazabilidad:** las cifras del baseline y del screening están registradas como resultados históricos de ejecuciones reales; ninguna celda debe presentar resultados nuevos si no se ejecutó el entrenamiento correspondiente.